In [1]:
!nvidia-smi

Mon Mar 23 14:12:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q ultralytics roboflow filterpy yt-dlp opencv-python-headless huggingface_hub hf_xet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 121.9 MB/s eta 0:00:00


In [3]:
!pip install -q ultralytics roboflow filterpy yt-dlp opencv-python-headless

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="-")
project = rf.workspace("tracker-qjlj1").project("drones_new")
dataset = project.version(4).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to DRONES_NEW-4 in yolov8:: 100%|██████████| 46008/46008 [00:06<00:00, 6798.85it/s] 


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import threading, zipfile, shutil
from pathlib import Path
from roboflow import Roboflow
from huggingface_hub import snapshot_download

# ── Roboflow ────────────────────────────────────────────
def get_roboflow():
    rf = Roboflow(api_key="-")
    project = rf.workspace("tracker-qjlj1").project("drones_new")
    global rf_dataset
    rf_dataset = project.version(4).download("yolov8")
    print("✅ Roboflow done")

# ── Seraphim ────────────────────────────────────────────
def get_seraphim():
    global seraphim_path
    seraphim_path = Path(snapshot_download(
        repo_id="lgrzybowski/seraphim-drone-detection-dataset",
        repo_type="dataset",
        local_dir="/content/seraphim_raw"
    ))
    zips = list(seraphim_path.rglob("*.zip"))
    print(f"Extracting {len(zips)} zips...")
    for zp in zips:
        with zipfile.ZipFile(zp, "r") as z:
            z.extractall(zp.parent)
        zp.unlink()
    print("✅ Seraphim done")

# Run both at the same time
t1 = threading.Thread(target=get_roboflow)
t2 = threading.Thread(target=get_seraphim)
t1.start(); t2.start()
t1.join(); t2.join()
print("🎉 Both datasets ready")

loading Roboflow workspace...
loading Roboflow project...


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

✅ Roboflow done
Extracting 7 zips...
✅ Seraphim done
🎉 Both datasets ready


In [6]:
import shutil, yaml
from pathlib import Path
from tqdm import tqdm

OUT = Path("/content/drone_dataset")
for split in ("train", "val", "test"):
    (OUT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUT / split / "labels").mkdir(parents=True, exist_ok=True)

# ── Seraphim (already class 0 = drone) ──────────────────
s_train_imgs = sorted((seraphim_path / "train" / "images").glob("*.*"))
split_idx = int(len(s_train_imgs) * 0.8)

for img in tqdm(s_train_imgs[:split_idx], desc="Seraphim → train"):
    lbl = seraphim_path / "train" / "labels" / (img.stem + ".txt")
    if not lbl.exists(): continue
    shutil.copy2(img, OUT / "train" / "images" / ("ser_" + img.name))
    shutil.copy2(lbl, OUT / "train" / "labels" / ("ser_" + img.stem + ".txt"))

for img in tqdm(s_train_imgs[split_idx:], desc="Seraphim → val"):
    lbl = seraphim_path / "train" / "labels" / (img.stem + ".txt")
    if not lbl.exists(): continue
    shutil.copy2(img, OUT / "val" / "images" / ("ser_" + img.name))
    shutil.copy2(lbl, OUT / "val" / "labels" / ("ser_" + img.stem + ".txt"))

for img in tqdm((seraphim_path / "test" / "images").glob("*.*"), desc="Seraphim → test"):
    lbl = seraphim_path / "test" / "labels" / (img.stem + ".txt")
    if not lbl.exists(): continue
    shutil.copy2(img, OUT / "test" / "images" / ("ser_" + img.name))
    shutil.copy2(lbl, OUT / "test" / "labels" / ("ser_" + img.stem + ".txt"))

# ── Roboflow drones_new ──────────────────────────────────
rf_path = Path(rf_dataset.location)
for rf_split, out_split in [("train","train"),("valid","val"),("test","test")]:
    src_img = rf_path / rf_split / "images"
    src_lbl = rf_path / rf_split / "labels"
    if not src_img.exists(): continue
    for img in tqdm(sorted(src_img.glob("*.*")), desc=f"Roboflow → {out_split}"):
        lbl = src_lbl / (img.stem + ".txt")
        if not lbl.exists(): continue
        shutil.copy2(img, OUT / out_split / "images" / ("rf_" + img.name))
        shutil.copy2(lbl, OUT / out_split / "labels" / ("rf_" + img.stem + ".txt"))

# ── data.yaml ────────────────────────────────────────────
cfg = {"path": str(OUT), "train": "train/images", "val": "val/images",
       "test": "test/images", "nc": 1, "names": ["drone"]}
with open(OUT / "data.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

# ── Summary ───────────────────────────────────────────────
for split in ("train", "val", "test"):
    n = len(list((OUT / split / "images").glob("*.*")))
    print(f"  {split}: {n:,} images")
print("✅ Merge complete")

Seraphim → val: 100%|██████████| 15027/15027 [00:30<00:00, 497.32it/s] 
Seraphim → test: 8349it [00:21, 379.72it/s]
Roboflow → test: 100%|██████████| 1063/1063 [00:00<00:00, 1563.76it/s]


  train: 80,363 images
  val: 16,706 images
  test: 9,412 images
✅ Merge complete


In [7]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="/content/drone_dataset/data.yaml",
    epochs=10,
    imgsz=640,
    batch=16,
    device=0,
    project="drone_runs",
    name="train1",
    exist_ok=True,
)

Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drone_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective

KeyboardInterrupt: 

In [8]:
import shutil, random, yaml
from pathlib import Path

random.seed(42)

# Sample 15k training images
src_img = Path("/content/drone_dataset/train/images")
src_lbl = Path("/content/drone_dataset/train/labels")
dst_img = Path("/content/drone_small/train/images")
dst_lbl = Path("/content/drone_small/train/labels")
dst_img.mkdir(parents=True, exist_ok=True)
dst_lbl.mkdir(parents=True, exist_ok=True)

imgs = random.sample(list(src_img.glob("*.*")), 15000)
for img in imgs:
    lbl = src_lbl / (img.stem + ".txt")
    shutil.copy2(img, dst_img / img.name)
    if lbl.exists():
        shutil.copy2(lbl, dst_lbl / lbl.name)

# Copy val and test as-is
for split in ("val", "test"):
    for sub in ("images", "labels"):
        shutil.copytree(
            f"/content/drone_dataset/{split}/{sub}",
            f"/content/drone_small/{split}/{sub}",
            dirs_exist_ok=True
        )

# Write data.yaml
cfg = {"path": "/content/drone_small", "train": "train/images",
       "val": "val/images", "test": "test/images", "nc": 1, "names": ["drone"]}
with open("/content/drone_small/data.yaml", "w") as f:
    yaml.dump(cfg, f)

print(f"✅ Subset ready — {len(imgs):,} train images")

✅ Subset ready — 15,000 train images


In [9]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="/content/drone_small/data.yaml",
    epochs=10,
    imgsz=640,
    batch=16,
    device=0,
    project="drone_runs",
    name="train1",
    exist_ok=True,
)

Ultralytics 8.4.25 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drone_small/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x788e9809cd70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [21]:
!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]" \
    -o "/content/videos/drone_video_1.mp4" \
    "https://www.youtube.com/watch?v=DhmZ6W1UAv4"

!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]" \
    -o "/content/videos/drone_video_2.mp4" \
    "https://www.youtube.com/watch?v=YrydHPwRelI"

[youtube] Extracting URL: https://www.youtube.com/watch?v=DhmZ6W1UAv4
[youtube] DhmZ6W1UAv4: Downloading webpage
[youtube] DhmZ6W1UAv4: Downloading android vr player API JSON
[info] DhmZ6W1UAv4: Downloading 1 format(s): 137+140
[download] Destination: /content/videos/drone_video_1.f137.mp4
[download] 100% of   10.89MiB in 00:00:00 at 26.63MiB/s
[download] Destination: /content/videos/drone_video_1.f140.m4a
[download] 100% of    2.56MiB in 00:00:00 at 26.38MiB/s
[Merger] Merging formats into "/content/videos/drone_video_1.mp4"
Deleting original file /content/videos/drone_video_1.f140.m4a (pass -k to keep)
Deleting original file /content/videos/drone_video_1.f137.mp4 (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=YrydHPwRelI
[youtube] YrydHPwRelI: Downloading webpage
[youtube] YrydHPwRelI: Downloading android vr player API JSON
[info] YrydHPwRelI: Downloading 1 format(s): 137+140
[download] Destination: /content/videos/drone_video_2.f137.mp4
[download] 100% o

In [22]:
import os
from pathlib import Path

videos_dir = Path("/content/videos")
frames_dir = Path("/content/frames")
frames_dir.mkdir(exist_ok=True)

for video in sorted(videos_dir.glob("*.mp4")):
    out_dir = frames_dir / video.stem
    out_dir.mkdir(exist_ok=True)
    os.system(
        f'ffmpeg -i "{video}" -vf "fps=5" '
        f'"{out_dir}/frame_%04d.jpg" -hide_banner -loglevel error'
    )
    n = len(list(out_dir.glob("*.jpg")))
    print(f"✅ {video.name} → {n} frames")

✅ drone_video_1.mp4 → 828 frames
✅ drone_video_2.mp4 → 2580 frames


In [23]:
from ultralytics import YOLO
from pathlib import Path
import shutil, cv2

# Load your trained weights
model = YOLO("/content/runs/detect/drone_runs/train1/weights/best.pt")

frames_dir  = Path("/content/frames")
detect_dir  = Path("/content/detections")
detect_dir.mkdir(exist_ok=True)

# Store per-video detections for the tracker
# Structure: {video_stem: {frame_idx: [(x1,y1,x2,y2,conf), ...]}}
all_detections = {}

for video_dir in sorted(frames_dir.iterdir()):
    if not video_dir.is_dir():
        continue

    frames = sorted(video_dir.glob("*.jpg"))
    vid_dets = {}
    saved = 0

    (detect_dir / video_dir.name).mkdir(exist_ok=True)

    for frame_path in frames:
        idx = int(frame_path.stem.split("_")[-1])
        results = model(str(frame_path), conf=0.25, verbose=False)[0]

        boxes = results.boxes
        if boxes is not None and len(boxes) > 0:
            dets = []
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = float(box.conf[0])
                dets.append((x1, y1, x2, y2, conf))
            vid_dets[idx] = dets

            # Save frame to detections/
            dst = detect_dir / video_dir.name / frame_path.name
            shutil.copy2(frame_path, dst)
            saved += 1
        else:
            vid_dets[idx] = []

    all_detections[video_dir.name] = vid_dets
    print(f"✅ {video_dir.name}: {saved}/{len(frames)} frames with detections")

✅ drone_video_1: 538/828 frames with detections
✅ drone_video_2: 334/2580 frames with detections


In [24]:
import cv2
import numpy as np
from filterpy.kalman import KalmanFilter
from pathlib import Path
from collections import defaultdict

# ── Kalman filter factory ────────────────────────────────────────────────────
def make_kalman():
    """
    State vector: [cx, cy, vx, vy]  (center x/y + velocity)
    Measurement:  [cx, cy]
    """
    kf = KalmanFilter(dim_x=4, dim_z=2)

    # State transition: constant velocity model
    kf.F = np.array([[1, 0, 1, 0],
                     [0, 1, 0, 1],
                     [0, 0, 1, 0],
                     [0, 0, 0, 1]], dtype=float)

    # Measurement matrix: observe cx, cy only
    kf.H = np.array([[1, 0, 0, 0],
                     [0, 1, 0, 0]], dtype=float)

    # Measurement noise (detector uncertainty)
    kf.R = np.eye(2) * 10.0

    # Process noise (motion uncertainty)
    kf.Q = np.eye(4) * 0.1
    kf.Q[2, 2] = 1.0   # higher uncertainty on velocity
    kf.Q[3, 3] = 1.0

    # Initial covariance
    kf.P = np.eye(4) * 50.0

    return kf


def box_center(x1, y1, x2, y2):
    return (x1 + x2) / 2, (y1 + y2) / 2


def best_detection(dets, predicted_cx, predicted_cy):
    """Pick detection closest to Kalman prediction."""
    best, best_dist = None, float("inf")
    for det in dets:
        cx, cy = box_center(*det[:4])
        dist = np.hypot(cx - predicted_cx, cy - predicted_cy)
        if dist < best_dist:
            best_dist = dist
            best = det
    return best, best_dist


# ── Per-video tracking ───────────────────────────────────────────────────────
MAX_MISS  = 5    # frames tracker keeps predicting without a detection
MAX_DIST  = 150  # pixels — max distance to associate detection with track

frames_dir = Path("/content/frames")
output_dir = Path("/content/output_videos")
output_dir.mkdir(exist_ok=True)

for video_stem, vid_dets in all_detections.items():
    print(f"\n🎯 Tracking {video_stem}…")

    frame_dir = frames_dir / video_stem
    frames    = sorted(frame_dir.glob("*.jpg"))
    if not frames:
        print("  No frames found, skipping.")
        continue

    # Read first frame for video dimensions
    sample = cv2.imread(str(frames[0]))
    h, w   = sample.shape[:2]

    out_path = output_dir / f"{video_stem}_tracked.mp4"
    writer   = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        5,          # same fps as extraction
        (w, h)
    )

    kf          = None
    miss_count  = 0
    initialized = False
    trajectory  = []        # list of (cx, cy) tracker estimates
    written     = 0

    for frame_path in frames:
        idx  = int(frame_path.stem.split("_")[-1])
        dets = vid_dets.get(idx, [])
        img  = cv2.imread(str(frame_path))

        # ── Initialize tracker on first detection ────────────────────────
        if not initialized:
            if not dets:
                continue
            best = max(dets, key=lambda d: d[4])  # highest conf
            cx, cy = box_center(*best[:4])
            kf = make_kalman()
            kf.x = np.array([[cx], [cy], [0.], [0.]])
            initialized = True
            miss_count  = 0
            trajectory.append((int(cx), int(cy)))

        # ── Predict ──────────────────────────────────────────────────────
        kf.predict()
        pred_cx = float(kf.x[0])
        pred_cy = float(kf.x[1])

        # ── Update ───────────────────────────────────────────────────────
        if dets:
            best_det, dist = best_detection(dets, pred_cx, pred_cy)
            if dist < MAX_DIST:
                cx, cy = box_center(*best_det[:4])
                kf.update(np.array([[cx], [cy]]))
                miss_count = 0
                draw_box = best_det
            else:
                miss_count += 1
                draw_box = None
        else:
            miss_count += 1
            draw_box = None

        # ── Stop tracking if too many misses ─────────────────────────────
        if miss_count > MAX_MISS:
            initialized = False
            trajectory  = []
            continue

        # ── Record tracker position ───────────────────────────────────────
        est_cx = int(float(kf.x[0]))
        est_cy = int(float(kf.x[1]))
        trajectory.append((est_cx, est_cy))

        # ── Draw bounding box ─────────────────────────────────────────────
        if draw_box is not None:
            x1, y1, x2, y2, conf = draw_box
            cv2.rectangle(img,
                          (int(x1), int(y1)), (int(x2), int(y2)),
                          (0, 255, 0), 2)
            cv2.putText(img, f"drone {conf:.2f}",
                        (int(x1), int(y1) - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        else:
            # Predicted position only — draw small circle
            cv2.circle(img, (est_cx, est_cy), 8, (0, 165, 255), -1)
            cv2.putText(img, "predicted",
                        (est_cx + 10, est_cy),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 2)

        # ── Draw trajectory polyline ──────────────────────────────────────
        if len(trajectory) > 1:
            for i in range(1, len(trajectory)):
                cv2.line(img, trajectory[i-1], trajectory[i],
                         (255, 0, 0), 2)

        writer.write(img)
        written += 1

    writer.release()
    print(f"  ✅ {written} frames written → {out_path}")

print("\n🎉 All tracking videos complete.")


🎯 Tracking drone_video_1…


Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)


  ✅ 564 frames written → /content/output_videos/drone_video_1_tracked.mp4

🎯 Tracking drone_video_2…
  ✅ 573 frames written → /content/output_videos/drone_video_2_tracked.mp4

🎉 All tracking videos complete.


In [25]:
from google.colab import files

for vid in sorted(Path("/content/output_videos").glob("*.mp4")):
    print(f"Downloading {vid.name}…")
    files.download(str(vid))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from pathlib import Path
from huggingface_hub import HfApi
import base64, os

rows = []
detect_dir = Path("/content/detections")

for video_dir in sorted(detect_dir.iterdir()):
    if not video_dir.is_dir():
        continue
    for img_path in sorted(video_dir.glob("*.jpg")):
        with open(img_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")
        rows.append({
            "video":    video_dir.name,
            "frame":    img_path.name,
            "image_b64": b64,
        })

df = pd.DataFrame(rows)
pq_path = "/content/detections.parquet"
df.to_parquet(pq_path, index=False)
print(f"✅ {len(df)} detection frames saved to parquet ({os.path.getsize(pq_path)/1e6:.1f} MB)")

# Upload to HuggingFace — replace with your HF username
api = HfApi()
api.upload_file(
    path_or_fileobj=pq_path,
    path_in_repo="detections.parquet",
    repo_id="naenile40/drone-detections",   # change this
    repo_type="dataset",
    token="-",                          # get at hf.co/settings/tokens
)
print("✅ Uploaded to HuggingFace")

✅ 872 detection frames saved to parquet (37.4 MB)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/detections.parquet :   3%|2         | 1.06MB / 37.4MB            

✅ Uploaded to HuggingFace
